<a href="https://colab.research.google.com/github/DevEnriquegd/mvp-nba/blob/main/mvp_nba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NBA Player Stats – Who’s the Real MVP?**

You are a Data Analyst for a sports media company. You’ve been given a dataset covering NBA players across many seasons with information on age, height, weight, draft position, and advanced stats (points, rebounds, assists, usage %, true shooting %, etc.).

Your task is to dig into this dataset to uncover trends in player performance, evaluate which metrics really define “greatness,” and nominate a “Data MVP” for a given season or all time.

In [18]:
import pandas as pd
from IPython.display import display
import numpy as np

url = 'https://raw.githubusercontent.com/DevEnriquegd/mvp-nba/7d874f9284eb1609c8e2eb08c243619fc560012b/all_seasons.csv'

data = pd.read_csv(url, index_col=0 )

data.head()

,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,draft_number,...,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
0,Randy Livingston,HOU,22.0,193.04,94.800728,Louisiana State,USA,1996,2,42,...,3.9,1.5,2.4,0.3,0.042,0.071,0.169,0.487,0.248,1996-97
1,Gaylon Nickerson,WAS,28.0,190.50,86.182480,Northwestern Oklahoma,USA,1994,2,34,...,3.8,1.3,0.3,8.9,0.030,0.111,0.174,0.497,0.043,1996-97
2,George Lynch,VAN,26.0,203.20,103.418976,North Carolina,USA,1993,1,12,...,8.3,6.4,1.9,-8.2,0.106,0.185,0.175,0.512,0.125,1996-97
3,George McCloud,LAL,30.0,203.20,102.058200,Florida State,USA,1989,1,7,...,10.2,2.8,1.7,-2.7,0.027,0.111,0.206,0.527,0.125,1996-97
4,George Zidek,DEN,23.0,213.36,119.748288,UCLA,USA,1995,1,22,...,2.8,1.7,0.3,-14.1,0.102,0.169,0.195,0.500,0.064,1996-97


In [19]:
# Data exploration - Basic Data

data[['player_name', 'age', 'player_height', 'player_weight', 'country', 'college']].sample(5)

,player_name,age,player_height,player_weight,country,college
2010,Derrick Dial,25.0,193.04,83.91452,USA,Eastern Michigan
11324,Cameron Oliver,24.0,203.20,102.05820,USA,Nevada
8340,Will Barton,24.0,198.12,79.37860,USA,Memphis
274,Stojko Vrankovic,33.0,218.44,117.93392,USA,NaN
2355,Steve Goodrich,26.0,208.28,99.79024,USA,Princeton


In [20]:
# Data exploration - Team and Draft

data[['player_name', 'team_abbreviation', 'season', 'draft_year', 'draft_round', 'draft_number']].sample(5)

,player_name,team_abbreviation,season,draft_year,draft_round,draft_number
12800,Jonathan Isaac,ORL,2022-23,2017,1,6
5051,Andre Iguodala,PHI,2007-08,2004,1,9
10762,Johnathan Williams,WAS,2019-20,Undrafted,Undrafted,Undrafted
4050,James Jones,PHX,2005-06,2003,2,49
12685,Justin Jackson,BOS,2022-23,2017,1,15


In [21]:
# Data exploration - Statistics Accumulated per Season

data[['player_name', 'season', 'gp', 'pts', 'reb', 'ast']].sample(5)

,player_name,season,gp,pts,reb,ast
7993,Marcin Gortat,2013-14,81,13.2,9.5,1.7
557,Tim Hardaway,1997-98,81,18.9,3.7,8.3
3537,Mickael Pietrus,2004-05,67,9.5,2.8,1.2
11898,Brandon Williams,2021-22,24,12.9,3.1,3.9
10875,Cody Martin,2019-20,48,5.0,3.3,2.0


In [22]:
# Data exploration - Statistics Accumulated per Season

data[['player_name', 'season', 'net_rating', 'oreb_pct', 'dreb_pct', 'usg_pct', 'ts_pct', 'ast_pct']].sample(5)

,player_name,season,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct
7870,Seth Curry,2013-14,-4.9,0.000,0.077,0.101,0.500,0.000
377,Matt Maloney,1996-97,6.3,0.010,0.064,0.147,0.585,0.188
3381,Tim Thomas,2003-04,0.7,0.034,0.140,0.224,0.534,0.102
9006,Kelly Olynyk,2015-16,5.2,0.054,0.158,0.207,0.561,0.119
2459,Keon Clark,2001-02,-0.8,0.095,0.232,0.214,0.522,0.071


In [23]:
# Data cleaning

data['college'].fillna('No College', inplace=True)

data.isnull().sum()

C:\Users\juan-\AppData\Local\Temp\ipykernel_6832\3156429046.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['college'].fillna('No College', inplace=True)


player_name          0
team_abbreviation    0
age                  0
player_height        0
player_weight        0
college              0
country              0
draft_year           0
draft_round          0
draft_number         0
gp                   0
pts                  0
reb                  0
ast                  0
net_rating           0
oreb_pct             0
dreb_pct             0
usg_pct              0
ts_pct               0
ast_pct              0
season               0
dtype: int64

## **Which players lead their seasons in scoring, rebounding, and playmaking - and how efficient are they?**

In [24]:
def find_leader_by_statistic(df, statistic):
    """
    Find the leader in a given statistic for each season in a DataFrame.
    """
    idx = df.groupby('season')[statistic].idxmax()
    return df.loc[idx]

report_cols = ['player_name', 'team_abbreviation', 'season', 'pts', 'reb', 'ast', 'net_rating', 'usg_pct', 'ts_pct', 'ast_pct']

In [25]:
leader_pts = find_leader_by_statistic(data, 'pts')

report_pts = leader_pts[report_cols].rename(columns={
    'pts': 'Leader_PTS',
    'reb': 'Reb_Context',
    'ast': 'Ast_Context'
})

efficient_scorers_report = report_pts.sort_values('ts_pct', ascending=False)

TS_ELITE_THRESHOLD = 0.60
TS_RISK_THRESHOLD = 0.53

def highlight_efficiency(val):
    """Assigns color based on the TS% value."""
    if isinstance(val, (int, float)):
        if val > TS_ELITE_THRESHOLD:
            return 'background-color: lightgreen'
        elif val < TS_RISK_THRESHOLD:
            return 'background-color: lightcoral'
    return ''

summary_cols = ['player_name', 'season', 'Leader_PTS', 'usg_pct', 'ts_pct', 'net_rating']

styled_report = efficient_scorers_report[summary_cols].style.format({
    'usg_pct': '{:.1%}',
    'ts_pct': '{:.1%}',
    'net_rating': '{:+.1f}'
}).map(
    highlight_efficiency,
    subset=['ts_pct']
)

print("🏀 SCORING LEADERS: RANKED BY TRUE SHOOTING EFFICIENCY (TS%)")

display(styled_report)

🏀 SCORING LEADERS: RANKED BY TRUE SHOOTING EFFICIENCY (TS%)


,player_name,season,Leader_PTS,usg_pct,ts_pct,net_rating
8930,Stephen Curry,2015-16,30.100000,32.0%,66.9%,+18.3
11537,Stephen Curry,2020-21,32.000000,33.1%,65.5%,+4.6
12839,Joel Embiid,2022-23,33.100000,37.0%,65.5%,+8.8
8013,Kevin Durant,2013-14,32.000000,32.7%,63.5%,+8.0
10634,James Harden,2019-20,34.300000,35.6%,62.6%,+5.8
9996,James Harden,2017-18,30.400000,35.3%,61.9%,+10.0
10227,James Harden,2018-19,36.100000,39.6%,61.6%,+6.3
12203,Joel Embiid,2021-22,30.600000,37.5%,61.6%,+7.9
6786,Kevin Durant,2011-12,28.000000,30.8%,61.0%,+7.7
6183,Kevin Durant,2009-10,30.100000,31.7%,60.7%,+7.0


In [26]:
leader_reb = find_leader_by_statistic(data, 'reb')

report_reb = leader_reb[report_cols].rename(columns={
    'reb': 'Leader_REB',
    'pts': 'Pts_Context',
    'ast': 'Ast_Context'
})

impact_rebounders_report = report_reb.sort_values('net_rating', ascending=False)
NET_RATING_ELITE_THRESHOLD = 8.0
NET_RATING_RISK_THRESHOLD = 0.0

def highlight_impact(val):
    """Assigns color based on the NET_RATING value."""
    if isinstance(val, (int, float)):
        if val >= NET_RATING_ELITE_THRESHOLD:
            return 'background-color: lightgreen'
        elif val < NET_RATING_RISK_THRESHOLD:
            return 'background-color: lightcoral'
    return ''

summary_cols_reb = ['player_name', 'season', 'Leader_REB', 'net_rating', 'usg_pct', 'ts_pct']

styled_report_reb = impact_rebounders_report[summary_cols_reb].style.format({
    'Leader_REB': '{:.1f}',
    'usg_pct': '{:.1%}',
    'ts_pct': '{:.1%}',
    'net_rating': '{:+.1f}'
}).map(
    highlight_impact,
    subset=['net_rating']
)

print("🧱 REBOUNDING LEADERS: RANKED BY NET RATING (Winning Impact)")

display(styled_report_reb)

🧱 REBOUNDING LEADERS: RANKED BY NET RATING (Winning Impact)


,player_name,season,Leader_REB,net_rating,usg_pct,ts_pct
188,Dennis Rodman,1996-97,16.1,+16.1,10.0%,47.9%
5971,Dwight Howard,2009-10,13.2,+11.8,24.1%,63.0%
8446,DeAndre Jordan,2014-15,15.0,+11.2,13.5%,63.8%
5445,Dwight Howard,2008-09,13.8,+10.6,26.0%,60.0%
3434,Kevin Garnett,2003-04,13.9,+10.4,29.4%,54.7%
12212,Rudy Gobert,2021-22,14.7,+9.6,16.6%,73.2%
7824,DeAndre Jordan,2013-14,13.6,+9.2,12.3%,63.0%
4918,Dwight Howard,2007-08,14.2,+8.1,24.0%,61.9%
760,Dennis Rodman,1997-98,15.0,+6.7,8.8%,45.9%
11411,Clint Capela,2020-21,14.3,+6.6,19.3%,60.1%


In [27]:
leader_ast = find_leader_by_statistic(data, 'ast')

report_ast = leader_ast[report_cols].rename(columns={
    'ast': 'Leader_AST',
    'pts': 'Pts_Context',
    'reb': 'Reb_Context'
})

creation_leaders_report = report_ast.sort_values('ast_pct', ascending=False)

AST_ELITE_THRESHOLD = 0.45
AST_RISK_THRESHOLD = 0.30

def highlight_creation(val):
    """Assigns color based on the AST% value."""
    if isinstance(val, (int, float)):
        if val >= AST_ELITE_THRESHOLD:
            return 'background-color: lightgreen'
        elif val < AST_RISK_THRESHOLD:
            return 'background-color: lightcoral'
    return ''

summary_cols_ast = ['player_name', 'season', 'Leader_AST', 'ast_pct', 'usg_pct', 'Pts_Context', 'net_rating']

styled_report_ast = creation_leaders_report[summary_cols_ast].style.format({
    'Leader_AST': '{:.1f}',
    'ast_pct': '{:.1%}',
    'usg_pct': '{:.1%}',
    'Pts_Context': '{:.1f}',
    'net_rating': '{:+.1f}'
}).map(
    highlight_creation,
    subset=['ast_pct']
)

print("🧠 ASSIST LEADERS: RANKED BY ASSIST PERCENTAGE (Playmaking Volume)")

display(styled_report_ast)

🧠 ASSIST LEADERS: RANKED BY ASSIST PERCENTAGE (Playmaking Volume)


,player_name,season,Leader_AST,ast_pct,usg_pct,Pts_Context,net_rating
5732,Chris Paul,2008-09,11.0,51.2%,27.3%,22.8,+6.5
9457,James Harden,2016-17,11.2,50.5%,34.1%,29.1,+6.3
5060,Chris Paul,2007-08,11.6,50.0%,25.4%,21.1,+7.9
6445,Steve Nash,2010-11,11.4,49.8%,21.2%,14.7,+4.5
7062,Rajon Rondo,2011-12,11.7,49.8%,20.3%,11.9,+5.2
7227,Rajon Rondo,2012-13,11.1,49.0%,21.5%,13.7,-1.3
2600,Andre Miller,2001-02,10.9,48.4%,22.7%,16.5,-1.1
5872,Steve Nash,2009-10,11.0,48.3%,22.8%,16.5,+7.1
10958,LeBron James,2019-20,10.2,47.7%,30.8%,25.3,+8.5
11555,Russell Westbrook,2020-21,11.7,47.7%,29.5%,22.2,-1.2


## **How do players from different eras (1990s, 2000s, 2010s, 2020s) compare in size, style, and performance??**

In [28]:
def assign_era(season):
    """
    Assigns a historical era (decade) based on the season string.
    Example: '1996-97' -> '1990s'
    """
    start_year = int(season[:4])

    if 1990 <= start_year <= 1999:
        return '1990s'
    elif 2000 <= start_year <= 2009:
        return '2000s'
    elif 2010 <= start_year <= 2019:
        return '2010s'
    elif 2020 <= start_year <= 2029:
        return '2020s'
    else:
        return 'Other'

In [29]:
data['era'] = data['season'].apply(assign_era)

data.head()

,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,draft_number,...,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season,era
0,Randy Livingston,HOU,22.0,193.04,94.800728,Louisiana State,USA,1996,2,42,...,1.5,2.4,0.3,0.042,0.071,0.169,0.487,0.248,1996-97,1990s
1,Gaylon Nickerson,WAS,28.0,190.50,86.182480,Northwestern Oklahoma,USA,1994,2,34,...,1.3,0.3,8.9,0.030,0.111,0.174,0.497,0.043,1996-97,1990s
2,George Lynch,VAN,26.0,203.20,103.418976,North Carolina,USA,1993,1,12,...,6.4,1.9,-8.2,0.106,0.185,0.175,0.512,0.125,1996-97,1990s
3,George McCloud,LAL,30.0,203.20,102.058200,Florida State,USA,1989,1,7,...,2.8,1.7,-2.7,0.027,0.111,0.206,0.527,0.125,1996-97,1990s
4,George Zidek,DEN,23.0,213.36,119.748288,UCLA,USA,1995,1,22,...,1.7,0.3,-14.1,0.102,0.169,0.195,0.500,0.064,1996-97,1990s


In [30]:
analysis_metrics = {
    'season': 'count',
    'player_height': 'mean',
    'player_weight': 'mean',
    'pts': 'mean',
    'reb': 'mean',
    'ast': 'mean',
    'net_rating': 'mean',
    'usg_pct': 'mean',
    'ts_pct': 'mean',
    'ast_pct': 'mean'
}

era_comparison = data.groupby('era').agg(analysis_metrics)

era_comparison = era_comparison.rename(columns={'season': 'Player_Count'})

print("\n--- 📈 Comparación de Promedios por Era ---")
display(era_comparison.sort_values(by='era'))


--- 📈 Comparación de Promedios por Era ---


,Player_Count,player_height,player_weight,pts,reb,ast,net_rating,usg_pct,ts_pct,ast_pct
era,,,,,,,,,,
1990s,1757,200.859693,100.541648,7.829653,3.534035,1.786511,-2.340751,0.188328,0.494335,0.133679
2000s,4469,201.037467,101.362800,8.102506,3.584247,1.782725,-2.149094,0.186365,0.502954,0.129900
2010s,4934,200.600422,100.014506,8.266133,3.550831,1.826429,-2.075598,0.184018,0.519172,0.131329
2020s,1684,198.824382,97.783823,8.747328,3.538064,1.970724,-2.753622,0.178043,0.542104,0.134698


## **Which teams, positions, or player types consistently produce top performers?**

In [31]:
metrics_to_normalize = ['pts', 'reb', 'ast', 'ts_pct', 'net_rating', 'dreb_pct']

mins = data[metrics_to_normalize].min()
maxs = data[metrics_to_normalize].max()
ranges = maxs - mins

for m in metrics_to_normalize:
    data[f'Norm_{m}'] = (data[m] - mins[m]) / ranges[m]


weights = {
    'pts': 0.30,
    'reb': 0.10,
    'ast': 0.10,
    'ts_pct': 0.20,
    'net_rating': 0.15,
    'dreb_pct': 0.15
}

data['MVP_Index'] = sum(
    data[f'Norm_{m}'] * w for m, w in weights.items()
)

H_SMALL = data['player_height'].quantile(0.30)
H_MEDIUM = data['player_height'].quantile(0.70)

h = data['player_height']
norm_ast = data['Norm_ast']
norm_reb = data['Norm_reb']

data['Inferred_Position'] = np.select(
    [
        (h <= H_SMALL) & (norm_ast > 0.6),    # PG
        (h <= H_SMALL),                       # SG
        (h < H_MEDIUM) & (norm_reb > 0.5),    # PF
        (h < H_MEDIUM),                       # SF
    ],
    ['PG', 'SG', 'PF', 'SF'],
    default='C'
)


USG_THRESHOLD = data['usg_pct'].quantile(0.6)
AST_THRESHOLD = data['ast_pct'].quantile(0.6)

is_high_usg = data['usg_pct'] >= USG_THRESHOLD
is_high_ast = data['ast_pct'] >= AST_THRESHOLD

data['player_type'] = np.select(
    [
        is_high_usg & is_high_ast,
        is_high_usg & ~is_high_ast,
        ~is_high_usg & is_high_ast
    ],
    [
        'Dual Threat / Balanced',
        'Scoring-Dominant',
        'Playmaking-Dominant'
    ],
    default='Role Player / Other'
)

MVP_THRESHOLD = data['MVP_Index'].quantile(0.95)
top_performers = data[data['MVP_Index'] >= MVP_THRESHOLD]

print(f"--- ✅ Se identificaron {len(top_performers)} Top Performers Históricos (MVP Index > {MVP_THRESHOLD:.4f}) ---\n")

teams_consistency = (
    top_performers.groupby('team_abbreviation')['player_name']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='Total_Top_Performers')
)

positions_consistency = (
    top_performers.groupby('Inferred_Position')['player_name']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='Total_Top_Performers')
)

types_consistency = (
    top_performers.groupby('player_type')['player_name']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='Total_Top_Performers')
)

print("🥇 Top 5 Teams (Consistent Producers of High Impact Players):")
print(teams_consistency.head(5))

print("\n📍 Top Inferred Positions (Consistent High Impact Roles):")
print(positions_consistency)

print("\n👤 Top Player Types (Dominant High Impact Roles):")
print(types_consistency)

--- ✅ Se identificaron 643 Top Performers Históricos (MVP Index > 0.4126) ---

🥇 Top 5 Teams (Consistent Producers of High Impact Players):
  team_abbreviation  Total_Top_Performers
0               LAL                    40
1               HOU                    29
2               MIN                    29
3               WAS                    28
4               MIA                    28

📍 Top Inferred Positions (Consistent High Impact Roles):
  Inferred_Position  Total_Top_Performers
0                 C                   284
1                SF                   137
2                SG                    98
3                PG                    94
4                PF                    30

👤 Top Player Types (Dominant High Impact Roles):
              player_type  Total_Top_Performers
0  Dual Threat / Balanced                   538
1        Scoring-Dominant                    95
2     Role Player / Other                     7
3     Playmaking-Dominant                     3


## **Based on the data, who deserves the MVP crown - and how does your pick compare to the official NBA MVP?**

In [32]:
idx_mvp_season = data.groupby('season')['MVP_Index'].idxmax()

mvp_by_season = data.loc[idx_mvp_season]

mvp_report_columns = [
    'season',
    'player_name',
    'team_abbreviation',
    'Inferred_Position',
    'MVP_Index',
    'pts',
    'reb',
    'ast',
    'ts_pct',
    'net_rating'
]

mvp_report = mvp_by_season[mvp_report_columns].sort_values(by='season', ascending=False)

print("--- 👑 MVP POR TEMPORADA (Basado en tu Índice Ponderado) ---")
display(mvp_report)

--- 👑 MVP POR TEMPORADA (Basado en tu Índice Ponderado) ---


,season,player_name,team_abbreviation,Inferred_Position,MVP_Index,pts,reb,ast,ts_pct,net_rating
12740,2022-23,Luka Doncic,DAL,PF,0.573943,32.4,8.6,8.0,0.609,2.1
12034,2021-22,Nikola Jokic,DEN,C,0.582948,27.1,13.8,7.9,0.661,8.4
11660,2020-21,Nikola Jokic,DEN,C,0.548687,26.4,10.8,8.3,0.647,7.7
10674,2019-20,Giannis Antetokounmpo,MIL,C,0.576616,29.5,13.6,5.6,0.613,15.4
10227,2018-19,James Harden,HOU,PG,0.580177,36.1,6.6,7.5,0.616,6.3
9671,2017-18,LeBron James,CLE,PF,0.540639,27.5,8.6,9.1,0.621,1.6
9309,2016-17,Russell Westbrook,OKC,PG,0.601935,31.6,10.7,10.4,0.554,3.3
8930,2015-16,Stephen Curry,GSW,SG,0.522555,30.1,5.4,6.7,0.669,18.3
8141,2014-15,Russell Westbrook,OKC,PG,0.517624,28.1,7.3,8.6,0.536,4.1
8013,2013-14,Kevin Durant,OKC,C,0.540966,32.0,7.4,5.5,0.635,8.0
